# PO Delay Root Cause Analyzer: Data Pipeline and EDA

**Objetivo de la semana:** Cargar, limpiar, validar y explorar el dataset de Purchase Orders. Entregar un DataFrame limpio con todos los deltas calculados, listo para la clasificación de la semana 2.

**Output final:** df_clean.csv y función reutilizable pipeline_core.py

---

### 1. Setup & Imports

In [ ]:
# Instalar dependencias
#%load_ext autoreload
#%autoreload 2
import os, warnings, matplotlib.pyplot as plt, matplotlib.ticker as mticker, numpy as np, pandas as pd, seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
from scipy import stats
#warnings.filterwarnings('ignore')
from pipeline_core import clean_po_data, cross_validate_deltas

# ESTILO GLOBAL DE GRÁFICAS
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#e8e8e8', 'grid.linewidth': 0.6, 'font.family': 'DejaVu Sans', 'axes.titlesize': 14, 'axes.titleweight': 'bold', 'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9, 'figure.dpi': 130})

# PALETAS DE COLORES CORPORATIVAS
PALETTE_MAIN, PALETTE_LATE, PALETTE_ONTIME, PALETTE_WARN = '#4C72B0', '#DD5C5C', '#55A868', '#F0A500'

print('✅ Setup completo')


## 2. Carga & Inventario de Columnas

## Descripción del Dataset

Para este análisis se utiliza **po_root_cause_synthetic.csv**, un conjunto de datos sintético diseñado para simular el ciclo de vida completo de las órdenes de compra (POs) dentro de una operación de retail. El dataset está compuesto por un volumen de **400 registros de POs únicos**, cuyos identificadores numéricos (`PO_NBR`) se encuentran en el rango del 100000 al 100399. Cabe destacar que todas las órdenes incluidas presentan un estado final de cerrado o completado (`PO_STATUS_CD = C`), lo que garantiza un ciclo de vida observable y analizable de principio a fin.

---

### Ecosistema Operativo

El entorno logístico del dataset involucra a múltiples actores y ubicaciones clave:
* **Proveedores:** Contempla la participación de **10 vendors**.
* **Centros de Distribución:** La distribución física se gestiona a través de **8 centros estratégicos**.
* **Transportistas:** El traslado de la mercancía está a cargo de una red de **9 empresas de transporte (*carriers*)**.

---

### Estructura y Variables Analíticas

A nivel técnico, el dataset cuenta con **39 columnas totales** que abarcan identificadores, marcas de tiempo (*timestamps*), cantidades, banderas (*flags*) y métricas de rendimiento.

La **variable objetivo** principal de este estudio es `IS_LATE (Y/N)`, la cual determina si una orden fue recibida con retraso respecto a su fecha estimada de llegada (`STA_DT`).

## Análisis de Valores Nulos

El inventario de datos faltantes revela dos escenarios operativos distintos: aquellos nulos que existen por diseño del negocio (esperados y correctos) y los nulos que se originan por fallas en el proceso de captura (problemáticos).

> **PREVIOUS_REQUEST_DT: 337 nulos (84.25%) — NULO POR DISEÑO**
>
> Esta variable solo registra un valor cuando una cita (*appointment*) ha sido explícitamente reprogramada en el sistema. Por lo tanto, la presencia de nulos es un comportamiento esperado y representa un **flag implícito de no reprogramación**.

> **TRAILER_ARRIVE_DT: 27 nulos (~6.75%) — NULO POR ERROR DE CAPTURA**
>
> Este indicador representa el problema de calidad de datos más crítico detectado en el repositorio. Al carecer del registro de la hora de llegada real del tráiler al patio de maniobras, resulta matemáticamente imposible clasificar si el retraso operativo fue responsabilidad del transportista (*carrier*) o del centro de distribución (DC). Como regla de negocio para el pipeline, estas 27 órdenes de compra (POs) deben marcarse de manera definitiva bajo la categoría de **"Unclassifiable"**.

---

### Incidencias Adicionales de Operación

Sumado a los puntos anteriores, la variable `REASON_DSC` presenta **9 valores nulos en total**, de los cuales **6 corresponden a órdenes de compra que llegaron tardías** (los 3 restantes son POs on-time, donde la ausencia de causa es esperada). Esos **6 casos** son los relevantes para el análisis: existe un retraso documentado por los timestamps en el ciclo logístico, pero no se capturó ninguna causa raíz en el sistema.

## 2.1 Carga del dataset (local)

In [ ]:
# Ejecución en local:

# DEFINICIÓN DE RUTA RAIZ
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "01_data_pipeline_and_eda":
    REPO_ROOT = REPO_ROOT.parent

load_dotenv(REPO_ROOT / ".env", override=True)

# RESOLVER RUTA AL CSV DESDE VARIABLE DE ENTORNO O USAR DEFAULT
_env_path = os.environ.get("PO_CSV_PATH") or None
CSV_PATH = Path(_env_path) if _env_path else REPO_ROOT / "data" / "raw" / "po_root_cause_synthetic.csv"

# VERIFICAR EXISTENCIA DEL CSV
if not CSV_PATH.exists():
    msg = (
        f"\n{'─'*60}\n"
        f"❌  CSV no encontrado\n\n"
        f"   Ruta buscada: {CSV_PATH}\n\n"
        f"   a) Coloca el archivo en data/raw/\n"
        f"      (ver data/raw/README.md)\n"
        f"   b) Define PO_CSV_PATH=/ruta/completa.csv en .env\n"
        f"{'─'*60}"
    )
    raise FileNotFoundError(msg)

# LEER CSV
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"✅ CSV cargado desde: {CSV_PATH}")
print(f"   Shape: {df_raw.shape}")

# 2. REPORTE DE CALIDAD (Diccionario en una sola línea)
null_report = pd.DataFrame({'dtype': df_raw.dtypes, 'nulls': df_raw.isnull().sum(), 'null_pct': (df_raw.isnull().sum() / len(df_raw) * 100).round(1), 'unique': df_raw.nunique()})
print('=== DIAGNÓSTICO DE COLUMNAS ===\n', null_report.to_string(), '\n\n=== PRIMERAS 5 FILAS DEL DATASET ===')

# 3. MUESTRA DE TABLA
display(df_raw.head(5))

## 3. Limpieza de Timestamps y Validación de Calidad

*Nota. Para no ser redundante, este bloque se explica mejor en el apartado de EDA y la descripción de la función reutilizable. De cualquier forma es importante dejar la salida con el resumen de resultados.*

In [ ]:
# Para revisar detalles de cálculos ir al punto 6. Función "clean_po_data()" — Output Reutilizable ´pipeline_core.py´

#1. EJECUTAR EL PIPELINE DE LIMPIEZA
df = clean_po_data(df_raw)

# 2. CONSTRUIR LA TABLA RESUMEN DE DIAGNÓSTICO
resumen_data = {
    "Métrica Operacional": [
        "Órdenes Totales Analizadas",
        "Órdenes con Datos Confiables (Sin errores)",
        "Órdenes con Errores de Cronología (Fechas invertidas)",
        "Órdenes sin Fecha de Llegada (TRAILER_ARRIVE_DT es Null)",
        "Entregas Reprogramadas (Cambio de cita)",
        "Órdenes con Faltante de Producto (Short Ship < 90%)",
        "Congestión en Patio (Espera > 4 hrs)",
        "Retraso en Andén de Carga (Dock Backlog > 6 hrs)"
    ],
    "Total Registros": [
        len(df),
        df["_data_reliable"].sum(),
        df["_ts_issue"].sum(),
        df["_trailer_arrive_null"].sum(),
        df["_rescheduled"].sum(),
        df["_short_ship"].sum(),
        df["flag_yard_congestion"].sum(),
        df["flag_dock_backlog"].sum()
    ]
}

# Crear DataFrame y calcular el porcentaje automáticamente
df_resumen = pd.DataFrame(resumen_data)
df_resumen["Porcentaje (%)"] = (df_resumen["Total Registros"] / len(df) * 100).round(1)

# Mostrar la tabla con formato estético
print("📊 RESUMEN EJECUTIVO LOGÍSTICO Y CALIDAD DE DATOS")
display(df_resumen.set_index("Métrica Operacional"))


## 4. Cálculo de Deltas y Métricas

*Nota. Para no ser redundante, este bloque se explica mejor en el apartado de EDA y la descripción de la función reutilizable. De cualquier forma es importante dejar la salida con el resumen de resultados.*

In [ ]:
# La limpieza y el cross-validation viven en pipeline_core.py (misma fuente que
# usa el script y que cubre el CI). Aqui solo invocamos; las tablas de abajo son
# presentacion de negocio sobre el df ya enriquecido, no re-calculo.

# 1. EJECUTAR EL PIPELINE DE LIMPIEZA
df = clean_po_data(df_raw)

# ── CROSS-VALIDATION: calculados vs columnas pre-calculadas ──────────────────
# cross_validate_deltas() imprime su reporte y anade _yard_discrepancy /
# _dock_discrepancy al df. Una sola fuente de verdad para la validacion.
df = cross_validate_deltas(df)

# Deltas absolutos para la tabla de presentacion (mismos que calcula la funcion)
delta_yard = (df['yard_wait_calc_hrs'] - df['YARD_WAIT_HRS']).abs()
delta_dock = (df['dock_calc_hrs'] - df['DOCK_HRS']).abs()
delta_delay = (df['delay_days_calc'] - df['DELAY_DAYS']).abs()

# Tabla de validacion de datos (vista de negocio)
df_cross_val = pd.DataFrame({
    "Métrica Evaluada": ["Espera en Patio (Horas)", "Procesamiento en Andén (Horas)", "Días de Retraso Total"],
    "Diferencia Promedio": [delta_yard.mean(), delta_dock.mean(), delta_delay.mean()],
    "Diferencia Máxima": [delta_yard.max(), delta_dock.max(), delta_delay.max()],
    "Alertas (> 1 hora de desfase)": [df['_yard_discrepancy'].sum(), df['_dock_discrepancy'].sum(), "N/A"]
})

print("🔍 1. REPORTE DE CONCORDANCIA DE DATOS (CROSS-VALIDATION)")
display(df_cross_val.set_index("Métrica Evaluada").round(3))


# ── DIAGNÓSTICO DE CUELLOS DE BOTELLA (FLAGS) ──────────────────────────────────
# ── Flags de clasificación preliminar ────────────────────────────────────
YARD_THRESHOLD_HRS = 4.0
DOCK_THRESHOLD_HRS = 6.0
CARRIER_MISS_HRS   = 4.0
SHORT_LEAD_DAYS    = 3.0
# Mapeamos las banderas técnicas generadas por la función a descripciones de negocio
flags_logistica = {
    "flag_yard_congestion": "Congestión en Patio (Espera > 4 horas)",
    "flag_dock_backlog": "Saturación en Andén (Procesamiento > 6 horas)",
    "flag_carrier_miss": "Incumplimiento de Transportista (Llegada > 4 horas tarde vs Cita)",
    "flag_short_lead_time": "Órdenes de Compra con Tiempo de Entrega Corto (< 3 días)",
    "flag_hot_late": "Órdenes Críticas (HOT) con Entrega Demorada"
}

df_cuellos_botella = pd.DataFrame({
    "Problema Detectado": flags_logistica.values(),
    "Órdenes Afectadas": [df[f].sum() for f in flags_logistica.keys()],
    "Impacto en el Dataset (%)": [(df[f].mean() * 100).round(1) for f in flags_logistica.keys()]
})

print("\n🚨 2. DIAGNÓSTICO AUTOMÁTICO DE CUELLOS DE BOTELLA")
display(df_cuellos_botella.set_index("Problema Detectado"))

## 5. EDA — Análisis exploratorio de Datos


### 5.1 Distribución Global de Delays

## Diagnóstico de Niveles de Retraso

El análisis revela una **tasa de entregas tardías del 41.25%**, un indicador críticamente elevado para cualquier operación de cadena de suministro en el sector retail. En los mercados comerciales modernos, el estándar de eficiencia se rige bajo la métrica *On Time In Full* (OTIF), la cual exige niveles de puntualidad superiores al 95%. Operar con casi la mitad de los pedidos fuera de tiempo genera un impacto financiero directo.

---

## Análisis de la Distribución de Retrasos (Días de Delay)

El análisis visual del histograma de **Días de retraso (solo POs tardíos)** revela un comportamiento operativo crítico: la distribución del delay no presenta una concentración en los primeros días ni una cola larga hacia la derecha. Por el contrario, muestra una **distribución notablemente simétrica y casi uniforme** en todo el espectro de tiempo analizado.

* **Concurrencia de Métricas Centrales:** La **Media (2.87 días)** y la **Mediana (2.90 días)** están prácticamente superpuestas en el centro del gráfico (cercanas a los 3 días). Esto confirma matemáticamente que los datos se distribuyen de manera equitativa a ambos lados del centro, descartando sesgos hacia retrasos cortos o largos.

### Implicaciones para la Cadena de Suministro y el Tablero

Esta simetría rompe la suposición común de que los retrasos grandes son eventos aislados o *outliers*. En este dataset, un retraso severo de **5 días es tan frecuente y probable** como un retraso leve de **1 día**.

Para la operación de supply chain, esto describe un proceso logístico **fuera de control estadístico**, donde el tiempo de desfase es impredecible.

Para el diseño del **dashboard**, esto cambia la estrategia:
1. No se deben buscar "casos aislados de alta severidad", ya que el volumen de afectación es masivo en todos los niveles de retraso.
2. El tablero debe enfocarse en **alertas tempranas de desviación estructural**, agrupando los impactos por transportista (*carrier*) o proveedor (*vendor*) para identificar si esta variabilidad constante es generalizada o provocada por actores específicos de la red.



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Distribución Global de Purchase Orders', fontsize=15, fontweight='bold', y=1.01)

# ── Plot 1: On-Time vs Late ──────────────────────────────────────────────────
ax = axes[0]
late_counts = df['IS_LATE'].value_counts()
colors = [PALETTE_LATE if x == 'Y' else PALETTE_ONTIME for x in late_counts.index]
bars = ax.bar(['On-Time\n(N)', 'Late\n(Y)'],
              [late_counts.get('N', 0), late_counts.get('Y', 0)],
              color=[PALETTE_ONTIME, PALETTE_LATE], width=0.5, edgecolor='white')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 3, f'{h}\n({h/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('On-Time vs Late')
ax.set_ylabel('Número de POs')
ax.set_ylim(0, 280)

# ── Plot 2: Distribución de DELAY_DAYS (solo tardíos) ───────────────────────
ax = axes[1]
late_df = df[df['IS_LATE'] == 'Y']
ax.hist(late_df['DELAY_DAYS'], bins=18, color=PALETTE_LATE, alpha=0.85, edgecolor='white')
ax.axvline(late_df['DELAY_DAYS'].mean(), color='#222', linestyle='--', linewidth=1.5,
           label=f'Media: {late_df["DELAY_DAYS"].mean():.2f}d')
ax.axvline(late_df['DELAY_DAYS'].median(), color='#888', linestyle=':', linewidth=1.5,
           label=f'Mediana: {late_df["DELAY_DAYS"].median():.2f}d')
ax.legend()
ax.set_title('Días de Delay (solo POs tardíos)')
ax.set_xlabel('Días de retraso')
ax.set_ylabel('Frecuencia')

# ── Plot 3: Reason Code distribution ────────────────────────────────────────
ax = axes[2]
reason_counts = df['REASON_DSC'].value_counts().drop('Not applicable', errors='ignore')
reason_counts = reason_counts.sort_values(ascending=True)
colors_reason = [PALETTE_WARN if 'Rescheduled' in r or 'Vendor' in r
                 else PALETTE_LATE if 'Carrier' in r or 'Missed' in r or 'Equipment' in r
                 else PALETTE_MAIN for r in reason_counts.index]
bars = ax.barh(reason_counts.index, reason_counts.values,
               color=colors_reason, edgecolor='white', height=0.6)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height()/2, str(int(w)),
            va='center', fontsize=9)
ax.set_title('Reason Codes (POs tardíos)')
ax.set_xlabel('Número de POs')

# Leyenda de colores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE_WARN,   label='Vendor/Scheduling'),
    Patch(facecolor=PALETTE_LATE,   label='Carrier/External'),
    Patch(facecolor=PALETTE_MAIN,   label='DC Operations'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

# Estadísticas descriptivas
print('── Estadísticas de DELAY_DAYS (todos los POs) ──────────────────────────')
print(df['DELAY_DAYS'].describe().round(3))
print(f'\nDelay medio (solo tardíos): {late_df["DELAY_DAYS"].mean():.2f} días')
print(f'POs con >3 días de retraso: {(df["DELAY_DAYS"]>3).sum()}')

### 5.2 Análisis por Distribution Center

## Análisis de Performance por Centro de Distribución (DCs)

Para evaluar el impacto de la infraestructura física en el cumplimiento de las órdenes de compra, se calcularon métricas de rendimiento de los centros de distribución.

---

### Hallazgos Estadísticos del Comportamiento Operativo

* **Foco Rojo en Severidad:** El centro de distribución de **Phoenix lidera el promedio general de días de retraso** de todo el dataset, consolidándose como la ubicación más crítica bajo la métrica `DELAY_DAYS`.
* **Heterogeneidad Estructural:** Existe disparidad en la eficiencia puntual entre los diferentes almacenes. La tasa de entregas tardías muestra una alta variabilidad geográfica, donde algunas ubicaciones superan el **45%** de incumplimiento mientras que los nodos más eficientes logran mantenerse por debajo del **35%**.
* **Desacople de Factores:** El centro de distribución que registra el mayor tiempo promedio de espera en patio no coincide necesariamente con aquel que presenta la peor tasa de entregas tardías. Este fenómeno indica que la saturación en los accesos exteriores no es la única causa del retraso final de la mercancía.
* **Eficiencia Interna en Andenes:** A nivel agregado, **ningún centro de distribución supera el umbral crítico de las 6.0 horas** en la zona de descarga. El backlog o saturación en los andenes (*dock*) queda descartado como el detonante principal de los retrasos a gran escala, limitándose únicamente a incidentes o valores atípicos (*outliers*) aislados.

---

### Interpretación Logística y Efecto Mix

La disparidad de rendimiento entre los centros de distribución es uno de los hallazgos más reveladores del estudio. Si todos los almacenes compartieran exactamente la misma cartera de proveedores (*vendors*) y transportistas (*carriers*), las variaciones de rendimiento serían atribuibles al 100% a la gestión interna de cada planta, como sus procesos de asignación de citas, turnos laborales o capacidad física de andenes. Un análisis más profundo de eso valdría la pena.

Ahora bien, dado que la distribución geográfica altera por completo el mix de socios comerciales asignados a cada región, una parte significativa del bajo desempeño puede estar ligada a factores externos al almacén.

El comportamiento crítico detectado en **Phoenix DC** ilustra perfectamente esta complejidad. Su posición como líder en días de retraso puede estar fundamentada en cuatro hipótesis distintas:
1. Una presión desproporcionada sobre sus recursos locales debido a un volumen de órdenes de compra significativamente más alto que el resto.
2. Un mix regional de proveedores con bajos niveles de cumplimiento en esa zona específica.
3. Distancias de tránsito de larga distancia más severas desde los puntos de manufactura u origen.
4. Ineficiencias puramente operativas en la administración interna del centro.

Por lo pronto, con estos hallazgos, el centro de distribución de **Phoenix se establece como un candidato elegible** para una auditoría de causa raíz profunda.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Performance por Distribution Center', fontsize=15, fontweight='bold')

dc_stats = df.groupby('DC_LOC_NAME').agg(
    total_pos=('PO_NBR', 'count'),
    late_pos=('IS_LATE', lambda x: (x == 'Y').sum()),
    avg_delay=('DELAY_DAYS', 'mean'),
    avg_yard=('YARD_WAIT_HRS', 'mean'),
    avg_dock=('DOCK_HRS', 'mean'),
    hot_late=('flag_hot_late', 'sum'),
).reset_index()
dc_stats['late_rate'] = dc_stats['late_pos'] / dc_stats['total_pos'] * 100
dc_stats = dc_stats.sort_values('late_rate', ascending=False)
dc_short = dc_stats['DC_LOC_NAME'].str.replace(' DC', '', regex=False)

# Plot 1: Late rate % por DC
ax = axes[0]
bar_colors = [PALETTE_LATE if r > 45 else PALETTE_WARN if r > 35 else PALETTE_ONTIME
              for r in dc_stats['late_rate']]
bars = ax.bar(dc_short, dc_stats['late_rate'], color=bar_colors, edgecolor='white')
ax.axhline(dc_stats['late_rate'].mean(), color='#444', linestyle='--', linewidth=1,
           label=f'Media nacional: {dc_stats["late_rate"].mean():.1f}%')
for bar, val in zip(bars, dc_stats['late_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Tasa de POs tardíos por DC')
ax.set_ylabel('% tardíos')
ax.set_ylim(0, 65)
ax.legend()
ax.tick_params(axis='x', rotation=30)

# Plot 2: Avg delay days por DC
ax = axes[1]
dc_s2 = dc_stats.sort_values('avg_delay', ascending=False)
ax.bar(dc_s2['DC_LOC_NAME'].str.replace(' DC', '', regex=False),
       dc_s2['avg_delay'], color=PALETTE_MAIN, edgecolor='white')
ax.axhline(dc_s2['avg_delay'].mean(), color='#444', linestyle='--', linewidth=1,
           label=f'Media: {dc_s2["avg_delay"].mean():.2f}d')
ax.set_title('Días promedio de delay por DC')
ax.set_ylabel('Días de retraso (promedio)')
ax.legend()
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('── Estadísticas por DC ─────────────────────────────────────────────────')
print(dc_stats.set_index('DC_LOC_NAME')[['total_pos','late_pos','late_rate','avg_delay','avg_yard','avg_dock']]
      .round(2).to_string())

### 5.3 Análisis por Vendor y Carrier

## Evaluación de Socios Comerciales: Análisis de Vendors y Carriers

La gestión contractual y la optimización de la cadena de suministro dependen críticamente de la visibilidad sobre los socios comerciales. Al aislar el desempeño de proveedores (*vendors*) y transportistas (*carriers*), el pipeline permite asignar responsabilidades con precisión y fundamentar las decisiones de negocio con datos objetivos.

---

### Rendimiento y Comportamiento de Vendors

El comportamiento operativo de los proveedores se evalúa bajo un enfoque bidimensional que cruza la puntualidad con el volumen de entrega, estableciendo reglas claras para la acción del negocio:

* **Tasa de Entregas Tardías (*Late Rate*):** Se observa un comportamiento  heterogéneo en la red, donde ciertos proveedores críticos superan el **50%** de incumplimiento. El umbral de alerta del negocio dictamina que cualquier indicador por encima de esta mitad de órdenes retrasadas exige una **acción contractual inmediata**.
* **Nivel de Surtido (*Fill Rate*):** El rendimiento fluctúa en valores promedio superiores al **90%**.

De acuerdo a los gráficos, el proveedor *MEDIQ* se consolida como el caso más grave de riesgo contractual. Presenta una **tasa de tardanza del 55%**, lo que significa que más de la mitad de sus camiones llegan fuera de su ventana, a pesar de que su Fill Rate roza el **98.5%** de efectividad.
* **Desviación Generalizada:** Siete de los diez proveedores superan la media general de retrasos (40.9%) o se encuentran en la zona de advertencia amarilla (por encima del 35%). Únicamente cuatro actores (*BIOPLEX, BIOMED, VITAGEN y AKZE*) muestran niveles de puntualidad moderadamente aceptables.

---

### Plan de Acción Contractual

Este cruce de datos redirige la atención hacia la **logística de transporte (*carriers*)**.

Si un proveedor como *MEDIQ* o *PRIMECARE* surte casi el 100% de las cajas pero llega tarde en la mitad de sus entregas, la causa raíz apunta de forma directa a dos escenarios: ineficiencias severas en el proceso de asignación y programación de citas (*scheduling*) o un desempeño deficiente de la empresa de transporte contratada para mover esa carga.

Para las mesas de negociación contractual, la estrategia ya no debe ser exigirles que fabriquen más rápido, sino **auditar las rutas de tránsito, revisar las ventanas horarias acordadas en el DC y evaluar el cambio de las empresas de transporte** asignadas a estas cuentas.

---

### Análisis de Eficiencia en Transportistas (Carriers)

El indicador de fallas del transportista (`flag_carrier_miss`) —definido como el porcentaje de órdenes donde la empresa de transporte arriba con un retraso mayor a las **4.0 horas** respecto a su cita aprobada— se consolida como el KPI definitivo para delimitar la responsabilidad del transporte en el desfase final de la red. Al analizar las 9 empresas de transporte del ecosistema, se detectó una variación en los niveles de cumplimiento de este indicador.

> **Costos Ocultos de la Operación en Patio**
>
> Un transportista con un índice elevado de citas perdidas (*carrier miss rate*) inyecta una serie de ineficiencias financieras que suelen ser invisibles para las métricas tradicionales de entrega (*On-Time Delivery*). Entre estos impactos destacan la necesidad de reprogramar muelles de urgencia, el pago de horas extra al personal operativo para recibir unidades desfasadas, la congestión física del patio de maniobras y un efecto cascada.

---

### Conclusiones de la Red de Socios

Los proveedores y transportistas que se ubican en el cuartil de peor rendimiento técnico quedan definidos como los objetivos prioritarios para la ejecución de acciones correctivas en el corto plazo.



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Performance por Vendor y Carrier', fontsize=15, fontweight='bold')

# ── Vendor stats ─────────────────────────────────────────────────────────────
vendor_stats = df.groupby('VENDOR_NAME').agg(
    total_pos=('PO_NBR', 'count'),
    late_pos=('IS_LATE', lambda x: (x == 'Y').sum()),
    avg_delay=('DELAY_DAYS', 'mean'),
    short_ship_pct=('_short_ship', 'mean'),
    avg_fill_rate=('_fill_rate', 'mean'),
).reset_index()
vendor_stats['late_rate'] = vendor_stats['late_pos'] / vendor_stats['total_pos'] * 100
vendor_stats = vendor_stats.sort_values('late_rate', ascending=False)

# ── Carrier stats ────────────────────────────────────────────────────────────
carrier_stats = df.groupby('CARRIER_PARTY_NAME').agg(
    total_pos=('PO_NBR', 'count'),
    late_pos=('IS_LATE', lambda x: (x == 'Y').sum()),
    avg_carrier_lag=('carrier_lag_hrs', 'mean'),
    carrier_miss_pct=('flag_carrier_miss', 'mean'),
).reset_index()
carrier_stats['late_rate'] = carrier_stats['late_pos'] / carrier_stats['total_pos'] * 100
carrier_stats = carrier_stats.sort_values('late_rate', ascending=False)

# Plot 1: Vendor late rate
ax = axes[0, 0]
bar_colors = [PALETTE_LATE if r > 50 else PALETTE_WARN if r > 35 else PALETTE_ONTIME
              for r in vendor_stats['late_rate']]
bars = ax.barh(vendor_stats['VENDOR_NAME'], vendor_stats['late_rate'],
               color=bar_colors, edgecolor='white', height=0.6)
ax.axvline(vendor_stats['late_rate'].mean(), color='#444', linestyle='--',
           linewidth=1.2, label=f'Media: {vendor_stats["late_rate"].mean():.1f}%')
for bar, val in zip(bars, vendor_stats['late_rate']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontsize=9)
ax.set_title('Tasa de POs tardíos por Vendor')
ax.set_xlabel('% tardíos')
ax.legend()

# Plot 2: Vendor fill rate (short ship)
ax = axes[0, 1]
vs2 = vendor_stats.sort_values('avg_fill_rate', ascending=True)
fill_colors = [PALETTE_LATE if r < 0.92 else PALETTE_WARN if r < 0.97 else PALETTE_ONTIME
               for r in vs2['avg_fill_rate']]
ax.barh(vs2['VENDOR_NAME'], vs2['avg_fill_rate'] * 100,
        color=fill_colors, edgecolor='white', height=0.6)
ax.axvline(90, color=PALETTE_LATE, linestyle='--', linewidth=1.2,
           label='Umbral short ship (90%)')
ax.set_title('Fill Rate promedio por Vendor')
ax.set_xlabel('Fill rate (%)')
ax.set_xlim(80, 102)
ax.legend()

# Plot 3: Carrier late rate
ax = axes[1, 0]
cs = carrier_stats.sort_values('late_rate', ascending=True)
ax.barh(cs['CARRIER_PARTY_NAME'], cs['late_rate'],
        color=PALETTE_MAIN, edgecolor='white', height=0.6)
ax.axvline(cs['late_rate'].mean(), color='#444', linestyle='--',
           linewidth=1.2, label=f'Media: {cs["late_rate"].mean():.1f}%')
ax.set_title('Tasa de POs tardíos por Carrier')
ax.set_xlabel('% tardíos')
ax.legend()

# Plot 4: Carrier miss % (llegó tarde a su ventana)
ax = axes[1, 1]
cs2 = carrier_stats.sort_values('carrier_miss_pct', ascending=True)
ax.barh(cs2['CARRIER_PARTY_NAME'], cs2['carrier_miss_pct'] * 100,
        color=PALETTE_WARN, edgecolor='white', height=0.6)
ax.set_title(f'% de Carrier Miss por Carrier\n(llegó >{CARRIER_MISS_HRS}h tarde a su ventana)')
ax.set_xlabel('% de POs con carrier miss')

plt.tight_layout()
plt.show()

print('── Top vendors por late rate ────────────────────────────────────────────')
print(vendor_stats[['VENDOR_NAME','total_pos','late_rate','avg_delay','avg_fill_rate']].round(3).to_string(index=False))

### 5.4 Reason Code Analysis + Mismatch Detection

## Análisis de Consistencia y Mismatches

Los datos analíticos revelan una vulnerabilidad crítica en la integridad del sistema de registro: **casi la mitad de los retrasos operativos están mal etiquetados**. Específicamente, el **48.5% de las órdenes de compra tardías** (80 de un universo de 165 POs retrasadas) presentan un código de razón (*Reason Code*) que es matemáticamente inconsistente frente a la evidencia cronológica de las marcas de tiempo reales (*timestamps*).

Este hallazgo trasciende el ámbito técnico de la limpieza de datos. Implica que la dirección de la empresa está tomando decisiones estratégicas en la cadena de suministro basadas en diagnósticos erróneos, lo que oculta ineficiencias internas detrás de justificaciones operativas registradas por el personal de piso.

> **Nota de método — qué cuenta como *mismatch* (y qué no):** esta clasificación por timestamps es **preliminar** (regla simple de la celda siguiente). De los 165 POs tardíos, **57 (≈35%) no disparan ninguna regla actual** y quedan como *"On-Time / Other"* — es decir, llegaron tarde pero la taxonomía vigente aún no los cubre. Parte del 48.5% es entonces **falta de cobertura de las reglas**, no necesariamente error humano. La cifra de mismatch "limpia" (solo donde una regla SÍ dispara y contradice al staff) se recalculará con el clasificador completo de la Semana 2.

---

Al evaluar la distribución de las inconsistencias por categoría, se identifica con claridad el origen de la fuga de información:
* **Concentración del Error:** Las etiquetas de *"Rescheduled by vendor"* (34 incidentes) y *"Vendor delayed shipment"* (28 incidentes) acaparan la inmensa mayoría de las contradicciones del sistema, sumando 62 de los 80 *mismatches* detectados.
* **El Insight Operativo:** El personal de compras o de administración del centro de distribución (DC) utiliza los códigos imputables al proveedor como un "comodín" o salida fácil ante incidencias. Mientras el sistema registra que el proveedor falló, las marcas de tiempo físicas demuestran que la mercancía o el transportista ya se encontraban listos en patio. Esta desconexión evidencia un vicio en el proceso de captura manual que castiga injustamente los indicadores de rendimiento (*Scorecards*) de los socios comerciales.

#### Radiografía Geográfica del Retraso
El análisis geográfico por centro de distribución permite mapear la severidad y el comportamiento de esta mala práctica en la red. Es importante distinguir **dos lecturas distintas** de la figura de al lado:

* **Volumen de la etiqueta (heatmap, izquierda):** la etiqueta *"Rescheduled by vendor"* es la más usada en POs tardíos, y se concentra en **Phoenix (PX, 13) y Dallas (DL, 13)**, seguidos de **Chicago (CH, 10) e Indianapolis (IN, 10)**. Esto mide *cuántas veces se usó la etiqueta*, no si fue correcta.
* **Mismatch real (subconjunto inconsistente):** de esos casos, los que **contradicen** los timestamps son menos — para *"Rescheduled by vendor"*: **PX=7, DL=7, CH=6**. Mirando **todas** las etiquetas, el mismatch total por DC lo lidera **Phoenix (21)**, seguido de **Kansas City (12), Dallas (11) y Chicago/otros**.
* **Patrón Sistemático de la Red:** en la totalidad de los nodos logísticos, las columnas correspondientes a fallas del proveedor concentran los valores más altos. Si el problema de puntualidad proviniera genuinamente de los proveedores, la distribución de los errores entre las distintas categorías de la red sería aleatoria. Al presentarse como una constante homogénea en todos los centros, se confirma que el problema es un síntoma de un vicio estructural en el proceso de recepción o de agendamiento general del negocio.

---

### Impacto Financiero y Operativo para el Negocio

* **Erosión del Poder de Negociación Contractual:** Al emitir penalizaciones y multas fundamentadas en datos erróneos (*mismatches*), los proveedores clave tienen el sustento técnico para rebatir las sanciones utilizando sus propios registros de telemetría y entrega. Esto debilita la postura legal y comercial de la compañía para exigir mejoras reales en los tiempos de entrega (*Lead Times*).

---

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('Reason Code Analysis y Detección de Mismatch', fontsize=15, fontweight='bold')

# ── Heatmap: DC vs Reason Code ───────────────────────────────────────────────
ax = axes[0]
late_only = df[df['IS_LATE'] == 'Y'].copy()
dc_reason = pd.crosstab(
    late_only['DC_FACILITY_CD_ABBREV'],
    late_only['REASON_DSC'].fillna('Sin código')
)
# Ordenar columnas por frecuencia total
dc_reason = dc_reason[dc_reason.sum().sort_values(ascending=False).index]

sns.heatmap(
    dc_reason, ax=ax, cmap='YlOrRd', annot=True, fmt='d',
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Número de POs'}
)
ax.set_title('Reason Code por DC\n(POs tardíos solamente)')
ax.set_xlabel('Reason Code')
ax.set_ylabel('DC')

# 👇 CORRECCIÓN AQUÍ: Rotación y alineación para evitar que se encimen las letras
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)

# ── Mismatch: reason code vs clasificación por timestamps ────────────────────
ax = axes[1]

# Regla simple de clasificación preliminar para comparar vs REASON_DSC
def classify_by_timestamps(row):
    if row['_rescheduled']:
        return 'Rescheduled'
    if row['flag_carrier_miss']:
        return 'Carrier'
    if row['flag_yard_congestion']:
        return 'Yard Congestion'
    if row['flag_dock_backlog']:
        return 'Dock Backlog'
    return 'On-Time / Other'

def simplify_reason(r):
    if pd.isna(r) or r == 'Not applicable':
        return 'On-Time / Other'
    if 'Rescheduled' in r:
        return 'Rescheduled'
    if 'Carrier' in r or 'Missed' in r or 'Equipment' in r or 'Weather' in r:
        return 'Carrier'
    if 'Yard' in r:
        return 'Yard Congestion'
    if 'Dock' in r:
        return 'Dock Backlog'
    if 'Vendor' in r:
        return 'Carrier'  # re-classify vendor as carrier-side for this exercise
    return 'On-Time / Other'

df['ts_classification'] = df.apply(classify_by_timestamps, axis=1)
df['reason_simplified'] = df['REASON_DSC'].apply(simplify_reason)

# Mismatch: timestamp classification != reason code classification
df['_mismatch'] = (
    (df['IS_LATE'] == 'Y') &
    (df['ts_classification'] != df['reason_simplified']) &
    (df['reason_simplified'] != 'On-Time / Other')
)

mismatch_n = df['_mismatch'].sum()
late_n = (df['IS_LATE'] == 'Y').sum()

# Visualizar mismatch por categoría de reason code
mismatch_by_reason = df[df['_mismatch']]['REASON_DSC'].value_counts()
bars = ax.barh(mismatch_by_reason.index, mismatch_by_reason.values,
               color=PALETTE_LATE, edgecolor='white', height=0.6)
for bar, val in zip(bars, mismatch_by_reason.values):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)

ax.set_title(f'Mismatch: Reason Code vs Timestamps\n'
             f'{mismatch_n} de {late_n} POs tardíos ({mismatch_n/late_n*100:.1f}%) '
             f'tienen reason code inconsistente')
ax.set_xlabel('Número de POs con mismatch')
ax.set_ylabel('Reason Code registrado por staff')

plt.tight_layout()
plt.show()

# Ejemplos concretos de mismatch
print('── Ejemplos de mismatch (top 5) ────────────────────────────────────────')
mismatch_examples = df[df['_mismatch']][[
    'PO_NBR','REASON_DSC','ts_classification',
    'YARD_WAIT_HRS','DOCK_HRS','carrier_lag_hrs','DELAY_DAYS'
]].head(5)
print(mismatch_examples.round(2).to_string(index=False))

## 6. Función "clean_po_data()" — Output Reutilizable


La función `clean_po_data` actúa como el motor principal para el procesamiento y transformación del conjunto de datos crudo (`df_raw`). Su objetivo fundamental no es solo limpiar inconsistencias técnicas, sino también realizar ingeniería de variables para generar métricas operacionales, cálculos de tiempos de espera (*deltas*) y banderas de alerta (*flags*) que faciliten el análisis de causa raíz.

El flujo interno de la función se divide estratégicamente en cinco etapas consecutivas:

---

### 1. Estandarización de Fechas y Tiempos
El primer paso consiste en homogeneizar el formato de **13 columnas críticas de marcas de tiempo** (*timestamps*). Variables esenciales como la fecha de creación de la orden (`PO_DT`), fechas de citas de entrega, y registros logísticos en el centro de distribución (como el arribo del tráiler o los procesos de *check-in* y *check-out*) se convierten explícitamente al tipo de dato `datetime`. En caso de encontrar cadenas corruptas o con formatos inválidos, se aplica una coerción para tratarlos de manera segura como nulos (`NaT`).

### 2. Flags de Calidad y Confiabilidad de Datos
Se construyen mecanismos de auditoría interna para evaluar la salud de la información mediante tres indicadores de control:
* **Identificación de nulos:** Se detecta de forma explícita si el arribo del transportista no fue registrado (`_trailer_arrive_null`).
* **Inconsistencias cronológicas:** Se activa la alerta `_ts_issue` si ocurre un error lógico temporal, como un *check-in* registrado antes de la llegada real del tráiler, o si la salida física es previa a su ingreso.
* **Filtro de confianza:** Se consolida el indicador `_data_reliable`, que etiqueta como verdaderamente confiables únicamente a aquellos registros que pasaron ambas auditorías con éxito.

### 3. Construcción de Métricas Operacionales
En este bloque se calculan indicadores de desempeño comercial y logístico:
* **Reprogramaciones (`_rescheduled`):** Compara la fecha de la cita aprobada actual contra la primera aprobada para rastrear cambios en la agenda.
* **Tasa de cumplimiento (`_fill_rate`):** Mide el porcentaje de cajas enviadas contra las solicitadas originalmente.
* **Faltante de mercancía (`_short_ship`):** Dispara una alerta si el *fill rate* de la orden cae por debajo del umbral crítico del **90%**.

### 4. Cálculo de Deltas y Tiempos Logísticos
Para cuantificar con precisión los cuellos de botella, la función calcula **7 nuevas métricas de tiempo**, convirtiéndolas dinámicamente a horas o días según corresponda:
* Tiempos de tránsito totales comprometidos en el puerto comercial (`lead_time_days`).
* El retraso de recolección de los transportistas tras la aprobación (`carrier_lag_hrs`).
* Tiempos de espera específicos en el patio de maniobras (`yard_wait_calc_hrs`) y andén de carga (`dock_calc_hrs`).
* Estancia total dentro del centro de distribución (`total_dc_hrs`).
*  Los días de anticipación con los que se programó la cita de entrega (`appt_lead_days`) y los días de retraso finales (`delay_days_calc`).

### 5. Clasificación por Etapa y Reglas de Negocio
Finalmente, se configuran umbrales operativos fijos para automatizar la detección y segmentación de problemas logísticos específicos:
* **Congestión de Patio (`flag_yard_congestion`):** Se activa al superar las **4.0 horas** de espera.
* **Atraso en Andén (`flag_dock_backlog`):** Se dispara si los trabajos de carga/descarga superan las **6.0 horas**.
* **Incumplimiento de Transportista (`flag_carrier_miss`):** Alerta cuando el desfase de recolección supera las **4.0 horas**.
* **Tiempo de Preparación Corto (`flag_short_lead_time`):** Identifica órdenes prioritarias o urgentes con menos de **3 días** de anticipación.
* **Órdenes Calientes Críticas (`flag_hot_late`):** Bandera combinada que aísla de inmediato aquellos pedidos etiquetados como de alta prioridad (*Hot*) que llegaron tarde.

---

### Validación de la Ejecución
Al finalizar el proceso, el script genera un reporte de control en la consola que certifica la correcta ejecución del pipeline. Este reporte muestra las dimensiones finales de la matriz (`Shape`), el volumen exacto de variables nuevas inyectadas al modelo y un listado explícito con el nombre de cada una de las columnas recién enriquecidas.


In [ ]:
# ── Verificación: el módulo produce el df enriquecido ────────────────────────
# clean_po_data() vive en pipeline_core.py (importada en la celda de setup). NO se
# redefine aquí: este bloque confirma que el import produce el mismo df_clean que
# el resto del notebook usa, como verificación visible de paridad con el módulo/CI.
df_clean = clean_po_data(df_raw)
df_clean = cross_validate_deltas(df_clean)

# ── Propagar el análisis de mismatch (celda 5.4) al df_clean ─────────────────
# clean_po_data() reconstruye df_clean desde cero, así que NO trae las columnas de
# mismatch (ts_classification / reason_simplified / _mismatch), que se calcularon
# sobre `df` en la celda de Reason Code Analysis. Las copiamos por POSICIÓN
# (.values) — ambos dataframes tienen las mismas 400 filas en el mismo orden — para
# que el resumen ejecutivo de abajo reporte el número real de mismatch (no "N/A") y
# para que el df_clean.csv exportado ya incluya la clasificación preliminar.
for _c in ["ts_classification", "reason_simplified", "_mismatch"]:
    if _c in df.columns:
        df_clean[_c] = df[_c].values

print('✅ clean_po_data() ejecutado correctamente (desde pipeline_core)')
print(f'   Shape:                      {df_clean.shape}')
print(f'   Columnas agregadas:         {df_clean.shape[1] - df_raw.shape[1]}')
print(f'   POs con datos confiables:   {df_clean["_data_reliable"].sum()} / {len(df_clean)}')
print(f'   POs rescheduled:            {df_clean["_rescheduled"].sum()}')
print(f'   Short ships:                {df_clean["_short_ship"].sum()}')
print(f'   Flag yard congestion:       {df_clean["flag_yard_congestion"].sum()}')
print(f'   Flag dock backlog:          {df_clean["flag_dock_backlog"].sum()}')
print(f'   Flag carrier miss:          {df_clean["flag_carrier_miss"].sum()}')

# Nuevas columnas
new_cols = [c for c in df_clean.columns if c not in df_raw.columns]
print(f'\nColumnas nuevas: {new_cols}')

## 7. Guardar Output y Resumen de Hallazgos

## 7.1 Guardar output y resumen de hallazgos (local)

In [ ]:
# ── Configurar Ruta de Salida Local en VSC ───────────────────────────────────
from pathlib import Path
import os

# Determinar la raíz del repositorio local de forma robusta
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "01_data_pipeline_and_eda":
    REPO_ROOT = REPO_ROOT.parent

# Definir y asegurar la existencia de la carpeta data/processed
OUTPUT_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ruta completa hacia el archivo final de salida
OUTPUT_CSV_PATH = OUTPUT_DIR / "df_clean.csv"

# ── Guardar DataFrame limpio ─────────────────────────────────────────────────
df_clean.to_csv(OUTPUT_CSV_PATH, index=False)
print('✅ df_clean.csv guardado exitosamente en tu repositorio local')
print(f'   Ruta: {OUTPUT_CSV_PATH}')
print(f'   Dimensiones: {df_clean.shape[0]} filas × {df_clean.shape[1]} columnas')

# ── Resumen ejecutivo ────────────────────────────────────────────────────────
late_df_c = df_clean[df_clean['IS_LATE'] == 'Y']

print()
print('='*70)
print('  RESUMEN EJECUTIVO — SEMANA 1')
print('='*70)

print(f'''
DATASET
  Total POs analizados:          {len(df_clean):>5}
  POs tardíos (IS_LATE=Y):       {(df_clean["IS_LATE"]=="Y").sum():>5}  ({(df_clean["IS_LATE"]=="Y").mean()*100:.1f}%)
  POs on-time (IS_LATE=N):       {(df_clean["IS_LATE"]=="N").sum():>5}  ({(df_clean["IS_LATE"]=="N").mean()*100:.1f}%)

CALIDAD DE DATOS
  POs con TRAILER_ARRIVE null:   {df_clean["_trailer_arrive_null"].sum():>5}  ← No clasificables
  POs con timestamps inválidos:  {df_clean["_ts_issue"].sum():>5}
  POs con datos confiables:      {df_clean["_data_reliable"].sum():>5}
  POs con reason code faltante:  {df_clean["REASON_DSC"].isna().sum() + (df_clean["REASON_CD"].isna() & (df_clean["IS_LATE"]=="Y")).sum():>5}

HALLAZGOS CLAVE
  Delay medio (tardíos):         {late_df_c["DELAY_DAYS"].mean():>5.2f} días
  Delay máximo:                  {df_clean["DELAY_DAYS"].max():>5.1f} días
  DC más problemático:           {df_clean.groupby("DC_LOC_NAME")["DELAY_DAYS"].mean().idxmax()}
  Vendor más tardío:             {df_clean.groupby("VENDOR_NAME")["IS_LATE"].apply(lambda x: (x=="Y").mean()).idxmax()}
  POs reprogramados:             {df_clean["_rescheduled"].sum():>5}  ({df_clean["_rescheduled"].mean()*100:.1f}%)
  Short ships (<90%):            {df_clean["_short_ship"].sum():>5}  ({df_clean["_short_ship"].mean()*100:.1f}%)
  HOT POs tardíos (críticos):    {df_clean["flag_hot_late"].sum():>5}

MISMATCH DETECTADO
  POs con reason code incorrecto: ~{df_clean["_mismatch"].sum() if "_mismatch" in df_clean.columns else "N/A"}  ← validar en semana 2

PRÓXIMOS PASOS (SEMANA 2)
  → Implementar clasificador completo por etapa (Vendor / Carrier / DC)
  → Calcular mismatch rate oficial con reglas de negocio completas
  → Preparar prompt template para LLM (semana 3)
''')

print('='*70)
print('  ARCHIVOS GENERADOS')
print('='*70)
# Se valida la existencia usando la ruta absoluta local en lugar de texto plano
exists = '✅' if OUTPUT_CSV_PATH.exists() else '❌'
print(f'  {exists} data/processed/df_clean.csv')
